In [ ]:
!git clone https://github.com/Atharvy700/SP-25.git


Cloning into 'SP-25'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 58 (delta 7), reused 48 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 8.44 MiB | 10.19 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [ ]:
%cd SP-25


/content/SP-25


In [ ]:
!ls -R


.:
bert_finetune.py  make_datasets.py  reflexivity.py  trans_eq_data
bert_noshot.py	  qwen_finetune.py  symmetry.py
datasets	  qwen_noshot.py    synthesis.py

./datasets:
reflexive  symmetric  transitive

./datasets/reflexive:
refl_test_iid.jsonl  refl_test_long.jsonl  refl_train.jsonl  refl_valid.jsonl

./datasets/symmetric:
sym_test_iid.jsonl  sym_test_long.jsonl  sym_train.jsonl  sym_valid.jsonl

./datasets/transitive:
eq_chain  mixed  negation  quantifiers	relations

./datasets/transitive/eq_chain:
trans_eq_test_iid.jsonl   trans_eq_train.jsonl
trans_eq_test_long.jsonl  trans_eq_valid.jsonl

./datasets/transitive/mixed:
mixed_test.jsonl  mixed_train.jsonl  mixed_valid.jsonl

./datasets/transitive/negation:
trans_neg_test.jsonl  trans_neg_train.jsonl  trans_neg_valid.jsonl

./datasets/transitive/quantifiers:
trans_quant_test.jsonl	trans_quant_train.jsonl  trans_quant_valid.jsonl

./datasets/transitive/relations:
trans_rel_test.jsonl  trans_rel_train.jsonl  trans_rel_valid.jsonl

./t

In [ ]:
# =====================================
# SmolLM3-3B — ZERO-SHOT INFERENCE
# =====================================

import json
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------------------------------
# CONFIG
# -------------------------------
MODEL_NAME = "HuggingFaceTB/SmolLM3-3B"
TEST_PATH = "trans_eq_data/test.json"   # expects [{"text": ..., "label": ...}]
BATCH_SIZE = 8
MAX_LENGTH = 512

# -------------------------------
# HELPERS
# -------------------------------
def normalize_label(label):
    if label is None:
        return None
    label = label.lower().strip()
    if label in ["yes", "true", "1"]:
        return "yes"
    if label in ["no", "false", "0"]:
        return "no"
    return None


def build_prompt(text):
    return (
        "Answer the following question with yes or no.\n\n"
        f"Question: {text}\n"
        "Answer:"
    )

# -------------------------------
# TOKENIZER
# -------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Safety: some small models have no pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# -------------------------------
# MODEL
# -------------------------------
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
model.eval()

# -------------------------------
# PREDICTION FUNCTION
# -------------------------------
@torch.no_grad()
def predict_batch(prompts):
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)

    outputs = model(**inputs)
    logits = outputs.logits[:, -1, :]  # last token logits

    yes_id = tokenizer.encode("yes", add_special_tokens=False)[0]
    no_id  = tokenizer.encode("no", add_special_tokens=False)[0]

    preds = []
    for logit in logits:
        preds.append("yes" if logit[yes_id] > logit[no_id] else "no")

    return preds

# -------------------------------
# LOAD DATA
# -------------------------------
with open(TEST_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

gold = [normalize_label(ex["label"]) for ex in data]
prompts = [build_prompt(ex["text"]) for ex in data]

# -------------------------------
# RUN ZERO-SHOT
# -------------------------------
preds = []
for i in range(0, len(prompts), BATCH_SIZE):
    batch_prompts = prompts[i:i + BATCH_SIZE]
    preds.extend(predict_batch(batch_prompts))

# -------------------------------
# EVALUATION
# -------------------------------
valid = [(p, g) for p, g in zip(preds, gold) if g is not None]
accuracy = np.mean([p == g for p, g in valid])

print("\n🎯 ZERO-SHOT ACCURACY:", round(float(accuracy), 4))
print("\nSample predictions:")
for i in range(min(5, len(valid))):
    print("GOLD:", valid[i][1], "| PRED:", valid[i][0])


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]